# Phase 5 — Legal Statute Identification (LSI): Experiment 3 = Phase 1 SWA Decoder + Bottleneck Adapters

**Technique.** Adapter tuning (scratch implementation, taken from Assignment 4): the Phase 1 **local/global sliding-window decoder** (16 layers, 1024-token context, window 128, global layers 4/8/12/16) is **completely frozen**. After the output of **every transformer block** a residual bottleneck adapter

$$h' = h + W_{up}\,\mathrm{GELU}(W_{down}h + b_{down}) + b_{up}$$

is inserted (`W_up`, `b_up` zero-initialised, so the model is *exactly* the frozen Phase 1 model at step 0). Only the adapters and the new classification head are trained.

**Adapting a causal decoder to multi-label classification**
* **1024-token input.** The model context is 1024, so each document is fed as *one* sequence of up to 1023 content tokens + `</s>`: the first 510 tokens (head) followed by cue-selected sentences from the rest of the document. This is the same "head + cue_rest" content that won in Exp 1 (two 512-token views), but now in a single sequence the model reads causally.
* **`</s>` at the end, included in attention pooling.** In a causal model only the last position has seen the whole input. Phase 1 pretraining appended `</s>` to every document, so the final `</s>` is an in-distribution "end of document" summary token. We append it and let the learned attention-pooling head attend over all content tokens **and** the `</s>` token. No `<s>` is prepended (Phase 1 never used it).

**No experiment sweep.** The classification recipe is the best one found in Exp 1: attention-pooling head, BCE with per-statute positive weights `sqrt(neg/pos)` clipped to [1,20], head bias initialised to the label prior, warm-up 6 % + linear decay, AMP fp16, post-processing chosen out-of-fold on dev. Only what has no Exp 1 counterpart (adapter width / adapter LR) is set once, by default, below.

**Final run:** full train set, **max 6 epochs, early stopping patience 2** on dev macro-F1, saves `.pt` + `submission.json`.

## 0. Configuration (edit this cell only)

In [1]:
import os, sys, re, gc, json, math, time, heapq, random, subprocess, warnings
from pathlib import Path
from dataclasses import dataclass, fields, asdict
from collections import OrderedDict

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore", category=UserWarning)

ON_KAGGLE = Path("/kaggle").exists()
WORK = Path("/kaggle/working") if ON_KAGGLE else Path("./lsi_work")
WORK.mkdir(parents=True, exist_ok=True)

# Phase 1 SWA decoder checkpoint. Either give the exact path, or leave "" and it is searched under /kaggle/input by file name.
PHASE1_CHECKPOINT_OVERRIDE = "/kaggle/input/datasets/vishwaspathania/swa-phase1/best_model_swa.pt"      # e.g. "/kaggle/input/datasets/<you>/swa-phase1/best_model_swa.pt"
PHASE1_CKPT_NAMES = ["best_model_swa.pt", "best_model.pt"]
PHASE1_TOKENIZER_OVERRIDE = "/kaggle/input/datasets/vishwaspathania/tokenizer-f/legal_bpe_a1_50k_final.json"

LOCAL_DATA_DIR = ""          # folder with train.json/dev.json/test.json, or "" to download from Hugging Face
HF_REPO_ID = "Exploration-Lab/CS-781-Capstone"
HF_FILES = {"train": "Phase_5_LSI/Train/train.json", "dev": "Phase_5_LSI/Train/dev.json", "test": "Phase_5_LSI/Test/test.json"}

NUM_LABELS = 100
MAX_LEN = 1024                # Phase 1 SWA context length (the Exp 1 encoder used 512)
CONTENT = MAX_LEN - 1         # content tokens per document; the last position is </s>
HEAD_KEEP = 510               # first 510 tokens are always kept (Exp 1 `head` view); the rest of the budget is cue-selected text
MAX_DOC_TOKENS = 16_384
TOK_CACHE_VERSION = "v1"      # same version as Exp 1 / Exp 2 -> the token cache is reused if it is still in /kaggle/working

# ---------------- adapter setting (scratch implementation from Assignment 4) ----------------
ADAPTER_BOTTLENECK = 64       # Assignment 4 used 16 on GPT-2; 64 gives more capacity for a frozen 100-label task (default, not tuned)
ADAPTER_INIT_STD = None       # None = exact-zero up-projection (model == frozen Phase 1 model at step 0), as in Assignment 4

# ---------------- classification recipe from Experiment 1 (no re-search) ----------------
RECIPE = dict(view="head+cue_rest (single 1024 sequence)", head="attn", loss="bce_posw", pool_eos=True,
              adapter_lr=1e-3, head_lr=1e-3, wd=0.01, dropout=0.1, warmup=0.06)
EFFECTIVE_BATCH = 32

# ---------------- final training ----------------
FINAL_MAX_EPOCHS = 6
FINAL_PATIENCE   = 2
RESUME_FINAL     = False      # True -> continue an interrupted run from final_last.pt
SMOKE_TEST       = False      # True -> tiny dry run of the whole pipeline
FINAL_TRAIN_DOCS_LIMIT = None
RUN_ADAPTER_ABLATION = True   # after training: switch adapters off layer-group by layer-group (no retraining) on a dev subset
ABLATION_DOCS = 600
EXP1_DEV_MACRO_F1 = 0.3826    # Exp 1 (encoder + full FT) best-epoch dev macro-F1 @ global threshold, for the comparison printout
SEED = 42

if SMOKE_TEST: FINAL_MAX_EPOCHS, FINAL_TRAIN_DOCS_LIMIT, ABLATION_DOCS = 2, 512, 128
RUN_DIR = WORK / ("lsi_exp3_swa_adapter_smoke" if SMOKE_TEST else "lsi_exp3_swa_adapter"); RUN_DIR.mkdir(parents=True, exist_ok=True)
CACHE_DIR = WORK / f"lsi_tok_cache_{TOK_CACHE_VERSION}"; CACHE_DIR.mkdir(parents=True, exist_ok=True)
print("WORK:", WORK.resolve(), "| ON_KAGGLE:", ON_KAGGLE)
print(f"adapter tuning: frozen = whole Phase 1 SWA decoder; trainable = 16 bottleneck adapters (width {ADAPTER_BOTTLENECK}) + attention-pooling head")

WORK: /kaggle/working | ON_KAGGLE: True
adapter tuning: frozen = whole Phase 1 SWA decoder; trainable = 16 bottleneck adapters (width 64) + attention-pooling head


## 1. Imports, GPU, seeds

In [2]:
import torch, torch.nn as nn, torch.nn.functional as F
from torch.utils.checkpoint import checkpoint
from torch.utils.data import Dataset, DataLoader
from tqdm.auto import tqdm
try:
    from IPython.display import display
except Exception:
    display = print

def seed_everything(s):
    random.seed(s); np.random.seed(s); torch.manual_seed(s)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(s)
seed_everything(SEED)
assert torch.cuda.is_available(), "Enable a GPU runtime."
device = torch.device("cuda")
gpu = torch.cuda.get_device_properties(0)
print("GPU:", gpu.name, f"| {gpu.total_memory/1024**3:.1f} GiB | torch {torch.__version__}")

GPU: Tesla T4 | 14.6 GiB | torch 2.11.0+cu128


## 2. Locate / download data and Phase 1 artifacts

In [4]:
try:
    import ijson
except ImportError:
    try:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "ijson"]); import ijson
    except Exception:
        ijson = None

def _get_hf_token():
    tok = os.environ.get("HF_TOKEN", "").strip()
    if not tok:
        try:
            from kaggle_secrets import UserSecretsClient
            tok = UserSecretsClient().get_secret("HF_TOKEN")
        except Exception:
            tok = ""
    if not tok:
        from getpass import getpass
        tok = getpass("Hugging Face token: ")
    return tok

def locate_data():
    paths = {}
    if LOCAL_DATA_DIR:
        for split in HF_FILES:
            p = Path(LOCAL_DATA_DIR) / f"{split}.json"
            if not p.is_file(): raise FileNotFoundError(p)
            paths[split] = p
        return paths
    from huggingface_hub import hf_hub_download
    token = _get_hf_token()
    for split, fn in HF_FILES.items():
        paths[split] = Path(hf_hub_download(repo_id=HF_REPO_ID, repo_type="dataset", filename=fn, token=token))
    return paths

def iter_json_records(path):
    with open(path, "rb") as f:
        first = b""
        while True:
            ch = f.read(1)
            if not ch: break
            if ch not in b" \t\r\n": first = ch; break
        f.seek(0)
        if first == b"[":
            if ijson is not None: yield from ijson.items(f, "item")
            else: yield from json.load(f)
        elif first == b"{":
            for line in f:
                line = line.strip()
                if line: yield json.loads(line)
        else:
            raise ValueError(f"Unrecognised JSON layout: {path}")

def find_artifact(filename, override=""):
    if override and Path(override).is_file(): return Path(override).resolve()
    hits = set()
    for root in [Path("/kaggle/input"), WORK, Path(".")]:
        if root.exists(): hits.update(p.resolve() for p in root.rglob(filename) if p.is_file())
    hits = sorted(hits)
    if len(hits) != 1: raise FileNotFoundError(f"{filename}: found {len(hits)} candidates {hits}. Set the override path in the config cell.")
    return hits[0]

need_raw = not all((CACHE_DIR / f"{s}_done.flag").exists() for s in ["train", "dev", "test"])
paths = locate_data() if need_raw else {}
def find_checkpoint():
    if PHASE1_CHECKPOINT_OVERRIDE and Path(PHASE1_CHECKPOINT_OVERRIDE).is_file(): return Path(PHASE1_CHECKPOINT_OVERRIDE).resolve()
    errs = []
    for n in PHASE1_CKPT_NAMES:
        try: return find_artifact(n)
        except FileNotFoundError as e: errs.append(str(e))
    raise FileNotFoundError("Phase 1 SWA checkpoint not found - set PHASE1_CHECKPOINT_OVERRIDE. " + " | ".join(errs))
PHASE1_CHECKPOINT_PATH = find_checkpoint()
PHASE1_TOKENIZER_PATH  = find_artifact("legal_bpe_a1_50k_final.json", PHASE1_TOKENIZER_OVERRIDE)
print("data:", {k: str(v) for k, v in paths.items()} if paths else "(token cache present, raw files not needed)")
print("checkpoint:", PHASE1_CHECKPOINT_PATH); print("tokenizer :", PHASE1_TOKENIZER_PATH)

Hugging Face token:  ········


data: {'train': '/root/.cache/huggingface/hub/datasets--Exploration-Lab--CS-781-Capstone/snapshots/93b0059be756c6c461f3a225de25aabda7d5f84a/Phase_5_LSI/Train/train.json', 'dev': '/root/.cache/huggingface/hub/datasets--Exploration-Lab--CS-781-Capstone/snapshots/93b0059be756c6c461f3a225de25aabda7d5f84a/Phase_5_LSI/Train/dev.json', 'test': '/root/.cache/huggingface/hub/datasets--Exploration-Lab--CS-781-Capstone/snapshots/93b0059be756c6c461f3a225de25aabda7d5f84a/Phase_5_LSI/Test/test.json'}
checkpoint: /kaggle/input/datasets/vishwaspathania/swa-phase1/best_model_swa.pt
tokenizer : /kaggle/input/datasets/vishwaspathania/tokenizer-f/legal_bpe_a1_50k_final.json


## 3. Phase 1 tokenizer and SWA decoder (scratch local/global GQA) + Assignment 4 adapters

In [5]:
SPECIAL_TOKENS = ["<pad>", "<unk>", "<s>", "</s>"]
N_SPECIAL, N_BYTES = 4, 256
BASE_VOCAB = N_SPECIAL + N_BYTES
WORD_RE = re.compile(r"\S+")

class A1BPETokenizer:
    def __init__(self, itos, merges):
        self.itos = list(itos); self.merges = [tuple(m) for m in merges]
        self.vocab_size = len(self.itos); self.stoi = {s: i for i, s in enumerate(self.itos)}
        self._byte_to_id = {bytes([b]): N_SPECIAL + b for b in range(256)}
        self._merge_ranks = {pair: r for r, pair in enumerate(self.merges)}
    def _encode_word(self, word):
        raw = word.encode("utf-8"); seq = [self._byte_to_id[bytes([b])] for b in raw]
        if len(seq) < 2 or not self._merge_ranks: return seq
        n = len(seq); prev = [-1] + list(range(n - 1)); nxt = list(range(1, n)) + [-1]
        active = [True] * n; heap = []
        for i in range(n - 1):
            r = self._merge_ranks.get((seq[i], seq[i + 1]))
            if r is not None: heapq.heappush(heap, (r, i, i + 1))
        while heap:
            rank, left, right = heapq.heappop(heap)
            if not active[left] or not active[right]: continue
            if nxt[left] != right or prev[right] != left: continue
            if self._merge_ranks.get((seq[left], seq[right])) != rank: continue
            seq[left] = BASE_VOCAB + rank; active[right] = False
            rn = nxt[right]; nxt[left] = rn
            if rn != -1: prev[rn] = left
            lp = prev[left]
            if lp != -1:
                nr = self._merge_ranks.get((seq[lp], seq[left]))
                if nr is not None: heapq.heappush(heap, (nr, lp, left))
            if rn != -1:
                nr = self._merge_ranks.get((seq[left], seq[rn]))
                if nr is not None: heapq.heappush(heap, (nr, left, rn))
        out, i = [], 0
        while i != -1:
            if active[i]: out.append(seq[i])
            i = nxt[i]
        return out

def load_a1_tokenizer(path):
    with open(path, "r", encoding="utf-8") as f: payload = json.load(f)
    tok = A1BPETokenizer(payload["itos"], payload["merges"])
    assert tok.vocab_size == 50_000 and tok.itos[:4] == SPECIAL_TOKENS
    return tok

phase1_tokenizer = load_a1_tokenizer(PHASE1_TOKENIZER_PATH)
PAD_ID, UNK_ID, BOS_ID, EOS_ID = 0, 1, 2, 3
SPACE_ID = N_SPECIAL + 32
_ckpt = torch.load(PHASE1_CHECKPOINT_PATH, map_location="cpu", weights_only=False)
PHASE1_STATE = dict(_ckpt["model_state_dict"]); _cfg_dict = dict(_ckpt["model_config"])
print("Phase 1 checkpoint:", {k: _ckpt.get(k) for k in ["global_step", "tokens_seen", "best_val_loss"]}); del _ckpt; gc.collect()

@dataclass
class ModelConfig:
    vocab_size: int = 50_000; d_model: int = 768; n_layers: int = 16; n_heads: int = 12
    n_kv_heads: int = 4; d_ff: int = 2048; max_seq_len: int = 1024
    local_window: int = 128; local_chunk_size: int = 256; global_layer_indices: tuple = (3, 7, 11, 15)
    rope_base: float = 10_000.0; rms_norm_eps: float = 1e-6; dropout: float = 0.0

cfg = ModelConfig(**{k: v for k, v in _cfg_dict.items() if k in {f.name for f in fields(ModelConfig)}})
cfg.global_layer_indices = tuple(cfg.global_layer_indices)
assert cfg.dropout == 0.0, "this notebook's attention has no dropout (Phase 1 used 0.0)"
assert MAX_LEN <= cfg.max_seq_len, f"MAX_LEN={MAX_LEN} exceeds the model context {cfg.max_seq_len}"
print(cfg)

def build_rope_cache(max_seq_len, d_head, base=10_000.0):
    inv_freq = 1.0 / (base ** (torch.arange(0, d_head // 2, dtype=torch.float32) * 2 / d_head))
    freqs = torch.outer(torch.arange(max_seq_len, dtype=torch.float32), inv_freq)
    ang = torch.cat([freqs, freqs], dim=-1)
    return ang.cos(), ang.sin()
def rotate_half(x):
    x1, x2 = torch.chunk(x, 2, dim=-1); return torch.cat([-x2, x1], dim=-1)
def apply_rope(x, cos, sin):
    return x * cos.to(x.device, x.dtype)[None, None] + rotate_half(x) * sin.to(x.device, x.dtype)[None, None]

class RMSNorm(nn.Module):
    def __init__(self, dim, eps=1e-6):
        super().__init__(); self.eps = eps; self.weight = nn.Parameter(torch.ones(dim))
    def forward(self, x):
        dt = x.dtype; xf = x.float()
        return (xf * torch.rsqrt(xf.pow(2).mean(-1, keepdim=True) + self.eps) * self.weight.float()).to(dt)

class SwiGLU(nn.Module):
    def __init__(self, d, ff):
        super().__init__()
        self.gate_proj = nn.Linear(d, ff, bias=False); self.up_proj = nn.Linear(d, ff, bias=False); self.down_proj = nn.Linear(ff, d, bias=False)
    def forward(self, x): return self.down_proj(F.silu(self.gate_proj(x)) * self.up_proj(x))

class CausalSelfAttention(nn.Module):
    """Phase 1 scratch GQA + RoPE. Local layers: sliding window (chunked batched matmul, exactly the Phase 1 code path).
    Global layers: full causal attention (same maths as Phase 1, computed with the fused SDPA kernel to save memory)."""
    def __init__(self, c, is_global):
        super().__init__()
        self.d_model, self.n_heads, self.n_kv_heads = c.d_model, c.n_heads, c.n_kv_heads
        self.head_dim = c.d_model // c.n_heads; self.n_rep = c.n_heads // c.n_kv_heads; self.max_seq_len = c.max_seq_len
        self.is_global, self.local_window, self.local_chunk = is_global, c.local_window, c.local_chunk_size
        self.q_proj = nn.Linear(c.d_model, c.n_heads * self.head_dim, bias=False)
        self.k_proj = nn.Linear(c.d_model, c.n_kv_heads * self.head_dim, bias=False)
        self.v_proj = nn.Linear(c.d_model, c.n_kv_heads * self.head_dim, bias=False)
        self.out_proj = nn.Linear(c.d_model, c.d_model, bias=False)
        cos, sin = build_rope_cache(c.max_seq_len, self.head_dim, c.rope_base)
        self.register_buffer("rope_cos", cos, persistent=False); self.register_buffer("rope_sin", sin, persistent=False)
    def _split(self, x, n):
        b, t, _ = x.shape; return x.reshape(b, t, n, self.head_dim).transpose(1, 2)
    def _global(self, q, k, v):
        return F.scaled_dot_product_attention(q, k.repeat_interleave(self.n_rep, dim=1), v.repeat_interleave(self.n_rep, dim=1), dropout_p=0.0, is_causal=True)
    def _local(self, q, k, v):
        B, Hq, T, D = q.shape; W = self.local_window; outs = []
        for s in range(0, T, self.local_chunk):
            e = min(s + self.local_chunk, T); ks = max(0, s - W + 1)
            qg = q[:, :, s:e].reshape(B, self.n_kv_heads, self.n_rep, e - s, D)
            kr = k[:, :, ks:e].unsqueeze(2); vr = v[:, :, ks:e].unsqueeze(2)
            sc = torch.matmul(qg, kr.transpose(-2, -1)) / math.sqrt(D)
            qp = torch.arange(s, e, device=q.device); kp = torch.arange(ks, e, device=q.device)
            valid = (kp[None, :] <= qp[:, None]) & (kp[None, :] >= qp[:, None] - W + 1)
            sc = sc.masked_fill(~valid[None, None, None], torch.finfo(sc.dtype).min)
            outs.append(torch.matmul(F.softmax(sc, dim=-1), vr).reshape(B, Hq, e - s, D))
        return torch.cat(outs, dim=2)
    def forward(self, x):
        b, t, _ = x.shape
        if t > self.max_seq_len: raise ValueError(f"sequence {t} > {self.max_seq_len}")
        q = apply_rope(self._split(self.q_proj(x), self.n_heads), self.rope_cos[:t], self.rope_sin[:t])
        k = apply_rope(self._split(self.k_proj(x), self.n_kv_heads), self.rope_cos[:t], self.rope_sin[:t])
        v = self._split(self.v_proj(x), self.n_kv_heads)
        o = self._global(q, k, v) if self.is_global else self._local(q, k, v)
        return self.out_proj(o.transpose(1, 2).contiguous().reshape(b, t, self.d_model))

class TransformerBlock(nn.Module):
    def __init__(self, c, is_global):
        super().__init__()
        self.norm1 = RMSNorm(c.d_model, c.rms_norm_eps); self.attn = CausalSelfAttention(c, is_global)
        self.norm2 = RMSNorm(c.d_model, c.rms_norm_eps); self.mlp = SwiGLU(c.d_model, c.d_ff)
    def forward(self, x):
        x = x + self.attn(self.norm1(x))
        return x + self.mlp(self.norm2(x))

class LegalBackbone(nn.Module):
    """Phase 1 LegalDecoderLM without the (tied) LM head: returns final-norm hidden states [B, T, d_model]."""
    def __init__(self, c, grad_ckpt=False):
        super().__init__(); self.config = c; self.grad_ckpt = grad_ckpt
        self.token_embedding = nn.Embedding(c.vocab_size, c.d_model)
        self.blocks = nn.ModuleList([TransformerBlock(c, i in c.global_layer_indices) for i in range(c.n_layers)])
        self.final_norm = RMSNorm(c.d_model, c.rms_norm_eps)
    def forward(self, input_ids):
        x = self.token_embedding(input_ids)
        for blk in self.blocks:
            x = checkpoint(blk, x, use_reentrant=False) if (self.grad_ckpt and self.training and torch.is_grad_enabled()) else blk(x)
        return self.final_norm(x)

def load_phase1_weights(bb, state):
    cleaned = {}
    for k, v in state.items():
        for pre in ("module.", "_orig_mod."):
            if k.startswith(pre): k = k[len(pre):]
        cleaned[k] = v
    cleaned.pop("lm_head.weight", None)          # tied to token_embedding.weight in Phase 1
    bb.load_state_dict(cleaned, strict=True); return bb

# ---- Assignment 4: residual bottleneck adapter (scratch) -------------------------------------------------------
class BottleneckAdapter(nn.Module):
    def __init__(self, hidden_size, bottleneck=16, init_std=None):
        super().__init__()
        self.down = nn.Linear(hidden_size, bottleneck); self.up = nn.Linear(bottleneck, hidden_size)
        if init_std is None:
            nn.init.zeros_(self.up.weight); nn.init.zeros_(self.up.bias)
        else:
            nn.init.normal_(self.down.weight, 0.0, init_std); nn.init.normal_(self.up.weight, 0.0, init_std)
            nn.init.zeros_(self.down.bias); nn.init.zeros_(self.up.bias)
        self.enabled = True
    def forward(self, h):
        if not self.enabled: return h
        return h + self.up(F.gelu(self.down(h)))

class AdapterBlock(nn.Module):
    """Wraps a transformer block: adapter is applied to the block output (between layer i and layer i+1)."""
    def __init__(self, block, adapter):
        super().__init__(); self.block = block; self.adapter = adapter
    def forward(self, x): return self.adapter(self.block(x))

def add_adapters(bb, bottleneck, init_std=None):
    for i in range(len(bb.blocks)):
        bb.blocks[i] = AdapterBlock(bb.blocks[i], BottleneckAdapter(bb.config.d_model, bottleneck, init_std))
    return bb

def adapter_param_count(d_model, n_layers, r):   # derived by hand: down (d*r + r) + up (r*d + d) per layer
    return n_layers * (2 * d_model * r + r + d_model)

_probe = LegalBackbone(cfg); load_phase1_weights(_probe, PHASE1_STATE)
BACKBONE_PARAMS = sum(p.numel() for p in _probe.parameters()); del _probe
print(f"Phase 1 SWA decoder verified: {BACKBONE_PARAMS:,} parameters (embedding counted once; LM head is tied)")
print(f"adapter parameters (width {ADAPTER_BOTTLENECK}, derived by hand): {adapter_param_count(cfg.d_model, cfg.n_layers, ADAPTER_BOTTLENECK):,}")
print("attention pattern:", " ".join("G" if i in cfg.global_layer_indices else "L" for i in range(cfg.n_layers)))

Phase 1 checkpoint: {'global_step': 15000, 'tokens_seen': 491683840, 'best_val_loss': 1.440508484840393}
ModelConfig(vocab_size=50000, d_model=768, n_layers=16, n_heads=12, n_kv_heads=4, d_ff=2048, max_seq_len=1024, local_window=128, local_chunk_size=256, global_layer_indices=(3, 7, 11, 15), rope_base=10000.0, rms_norm_eps=1e-06, dropout=0.0)
Phase 1 SWA decoder verified: 139,088,640 parameters (embedding counted once; LM head is tied)
adapter parameters (width 64, derived by hand): 1,586,176
attention pattern: L L L G L L L G L L L G L L L G


### 3b. Self-tests (run once, ~seconds): the pieces below must hold before any GPU hours are spent
* local-chunked and global attention equal naive masked softmax attention (across chunk boundaries)
* the decoder is causal and right-padding does not change real positions
* adapters: step-0 identity with the frozen model, `enabled=False` restores it exactly, gradients reach every adapter and nothing else, parameter count equals the hand formula

In [6]:
def _run_self_tests():
    torch.manual_seed(0); dev = device
    sc = ModelConfig(vocab_size=128, d_model=64, n_layers=4, n_heads=4, n_kv_heads=2, d_ff=96, max_seq_len=64, local_window=8, local_chunk_size=16, global_layer_indices=(1, 3))
    B, T, D = 2, 50, sc.d_model // sc.n_heads
    q = torch.randn(B, sc.n_heads, T, D, device=dev); k = torch.randn(B, sc.n_kv_heads, T, D, device=dev); v = torch.randn(B, sc.n_kv_heads, T, D, device=dev)
    kr, vr = k.repeat_interleave(sc.n_heads // sc.n_kv_heads, 1), v.repeat_interleave(sc.n_heads // sc.n_kv_heads, 1)
    pos = torch.arange(T, device=dev); causal = pos[None, :] <= pos[:, None]; band = causal & (pos[None, :] >= pos[:, None] - sc.local_window + 1)
    def naive(mask):
        s = (q @ kr.transpose(-2, -1)) / math.sqrt(D); return torch.softmax(s.masked_fill(~mask, float("-inf")), -1) @ vr
    att_l = CausalSelfAttention(sc, False).to(dev); att_g = CausalSelfAttention(sc, True).to(dev)
    e_l = (att_l._local(q, k, v) - naive(band)).abs().max().item(); e_g = (att_g._global(q, k, v) - naive(causal)).abs().max().item()
    assert e_l < 1e-4 and e_g < 1e-4, f"attention mismatch local {e_l:.2e} global {e_g:.2e}"
    base = LegalBackbone(sc).to(dev).eval(); ids = torch.randint(4, 128, (3, 40), device=dev)
    with torch.no_grad():
        h0 = base(ids); ids2 = ids.clone(); ids2[:, 25:] = torch.randint(4, 128, (3, 15), device=dev)
        causal_err = (h0[:, :25] - base(ids2)[:, :25]).abs().max().item()
        ids3 = ids.clone(); ids3[:, 30:] = PAD_ID; pad_err = (base(ids3)[:, :30] - base(ids[:, :30])).abs().max().item()
    assert causal_err < 1e-5, f"model is not causal ({causal_err:.2e})"; assert pad_err < 1e-4, f"right padding changes real positions ({pad_err:.2e})"
    for p in base.parameters(): p.requires_grad_(False)
    r = 8; add_adapters(base, r); base.to(dev)
    with torch.no_grad(): ident = (base(ids) - h0).abs().max().item()
    assert ident == 0.0, f"zero-init adapters are not an exact identity ({ident:.2e})"
    base.train(); base(ids).pow(2).mean().backward()
    for i, blk in enumerate(base.blocks):
        assert blk.adapter.up.weight.grad is not None and blk.adapter.up.weight.grad.abs().sum() > 0, f"no gradient reaches adapter {i}"
    assert all(p.grad is None for n, p in base.named_parameters() if ".adapter." not in n), "a frozen parameter received a gradient"
    assert sum(p.numel() for p in base.parameters() if p.requires_grad) == adapter_param_count(sc.d_model, sc.n_layers, r), "adapter parameter count != hand formula"
    base.eval()
    for blk in base.blocks: nn.init.normal_(blk.adapter.up.weight, 0, 0.5)
    with torch.no_grad():
        moved = (base(ids) - h0).abs().max().item()
        for blk in base.blocks: blk.adapter.enabled = False
        restored = (base(ids) - h0).abs().max().item()
    assert moved > 1e-3 and restored == 0.0, f"enabled switch broken (moved {moved:.2e}, restored {restored:.2e})"
    print(f"self-tests passed | attention err local {e_l:.1e} / global {e_g:.1e} | causal {causal_err:.1e} | padding {pad_err:.1e} | adapter identity {ident:.0f} | all-off restore exact")
_run_self_tests()

self-tests passed | attention err local 4.8e-07 / global 4.8e-07 | causal 0.0e+00 | padding 7.2e-07 | adapter identity 0 | all-off restore exact


## 4. Token cache (full documents, tokenised once; reused from Exp 1 / Exp 2 if present)

In [7]:
CUE_RE = re.compile(r"<(?:SECTION|ACT)>|^(?:[Ss]ections?|[Ss]ec\.?|[Aa]rticles?|[Rr]ules?|[Aa]ct|[Aa]cts|IPC|CrPC|Cr\.P\.C\.?|Code)\W*$")
SENT_END_CHARS = (".", ";", "?", "!", ":")
FLAG_CUE, FLAG_SENT = 1, 2

class FastDocTokenizer:
    def __init__(self, tok, max_doc_tokens=MAX_DOC_TOKENS, tail_keep=4096, cache_limit=3_000_000):
        self.tok, self.max_doc_tokens, self.tail_keep, self.cache_limit = tok, max_doc_tokens, tail_keep, cache_limit; self.cache = {}
    def _word(self, w):
        r = self.cache.get(w)
        if r is None:
            ids = self.tok._encode_word(w); fl = [0] * len(ids)
            if CUE_RE.search(w): fl[0] |= FLAG_CUE
            if w.endswith(SENT_END_CHARS): fl[-1] |= FLAG_SENT
            r = (tuple(ids), tuple(fl))
            if len(self.cache) > self.cache_limit: self.cache.clear()
            self.cache[w] = r
        return r
    def encode(self, text):
        ids, fl, first = [], [], True
        for m in WORD_RE.finditer(text):
            if not first: ids.append(SPACE_ID); fl.append(0)
            w_ids, w_fl = self._word(m.group(0)); ids.extend(w_ids); fl.extend(w_fl); first = False
        n = len(ids)
        if n > self.max_doc_tokens:
            cut = self.max_doc_tokens - self.tail_keep
            ids = ids[:cut] + ids[-self.tail_keep:]; fl = fl[:cut] + fl[-self.tail_keep:]
        return np.asarray(ids, dtype=np.uint16), np.asarray(fl, dtype=np.uint8), n

_WORKER = {}
def _init_worker(): _WORKER["t"] = FastDocTokenizer(phase1_tokenizer)
def _tok_batch(texts):
    t = _WORKER["t"]; out = []
    for s in texts:
        i, f, n = t.encode(s); out.append((i, f, n))
    return out

def build_split_cache(split):
    flag = CACHE_DIR / f"{split}_done.flag"
    if flag.exists(): return
    has_labels = split != "test"; ids_raw, texts, labels = [], [], []
    for row in tqdm(iter_json_records(paths[split]), desc=f"read {split}"):
        ids_raw.append(row["id"]); texts.append(row["text"])
        if has_labels:
            lab = sorted(set(int(x) for x in row["labels"])); assert all(0 <= x < NUM_LABELS for x in lab); labels.append(lab)
    n_docs = len(texts); batches = [texts[i:i + 128] for i in range(0, n_docs, 128)]; results = []
    try:
        import multiprocessing as mp
        ctx = mp.get_context("fork"); nproc = min(4, os.cpu_count() or 1)
        with ctx.Pool(nproc, initializer=_init_worker) as pool:
            for r in tqdm(pool.imap(_tok_batch, batches), total=len(batches), desc=f"tokenise {split}"): results.extend(r)
    except Exception as e:
        print("parallel tokenisation unavailable (", type(e).__name__, ") -> sequential"); _init_worker(); results = []
        for b in tqdm(batches, desc=f"tokenise {split} (seq)"): results.extend(_tok_batch(b))
    lengths = np.array([len(r[0]) for r in results], dtype=np.int64)
    offsets = np.zeros(n_docs + 1, dtype=np.int64); offsets[1:] = np.cumsum(lengths)
    np.save(CACHE_DIR / f"{split}_tokens.npy", np.concatenate([r[0] for r in results]))
    np.save(CACHE_DIR / f"{split}_flags.npy", np.concatenate([r[1] for r in results]))
    np.save(CACHE_DIR / f"{split}_offsets.npy", offsets)
    np.save(CACHE_DIR / f"{split}_orig_len.npy", np.array([r[2] for r in results], dtype=np.int64))
    if has_labels:
        Y = np.zeros((n_docs, NUM_LABELS), dtype=np.uint8)
        for i, lab in enumerate(labels): Y[i, lab] = 1
        np.save(CACHE_DIR / f"{split}_labels.npy", Y)
    with open(CACHE_DIR / f"{split}_case_ids.json", "w", encoding="utf-8") as f: json.dump(ids_raw, f)
    flag.write_text("ok"); del results, texts; gc.collect()
    print(f"{split}: {n_docs:,} docs | {lengths.sum():,} tokens")

t0 = time.time()
for s in ["train", "dev", "test"]: build_split_cache(s)
print(f"token cache ready in {(time.time()-t0)/60:.1f} min -> {CACHE_DIR}")

def load_arrays(split, has_labels=True):
    d = dict(tokens=np.load(CACHE_DIR / f"{split}_tokens.npy", mmap_mode="r"), flags=np.load(CACHE_DIR / f"{split}_flags.npy", mmap_mode="r"),
             offsets=np.load(CACHE_DIR / f"{split}_offsets.npy"), orig_len=np.load(CACHE_DIR / f"{split}_orig_len.npy"))
    d["ids"] = json.load(open(CACHE_DIR / f"{split}_case_ids.json"))
    if has_labels: d["labels"] = np.load(CACHE_DIR / f"{split}_labels.npy")
    return d

ARR = {"train": load_arrays("train"), "dev": load_arrays("dev"), "test": load_arrays("test", has_labels=False)}
Ytr, Ydv = ARR["train"]["labels"], ARR["dev"]["labels"]
PRIOR = np.clip(Ytr.mean(0), 1e-5, 1 - 1e-5); POS_COUNT = Ytr.sum(0).astype(np.float64)
PRIOR_LOGIT = np.log(PRIOR / (1 - PRIOR)).astype(np.float32)
Ntr, Ndv, Nte = len(Ytr), len(Ydv), len(ARR["test"]["ids"])
print(f"train {Ntr:,} | dev {Ndv:,} | test {Nte:,} | labels/doc {Ytr.sum(1).mean():.2f}")

read train: 0it [00:00, ?it/s]

tokenise train:   0%|          | 0/334 [00:00<?, ?it/s]

train: 42,750 docs | 102,823,711 tokens


read dev: 0it [00:00, ?it/s]

tokenise dev:   0%|          | 0/80 [00:00<?, ?it/s]

dev: 10,181 docs | 27,323,268 tokens


read test: 0it [00:00, ?it/s]

tokenise test:   0%|          | 0/102 [00:00<?, ?it/s]

test: 13,019 docs | 33,541,917 tokens
token cache ready in 0.9 min -> /kaggle/working/lsi_tok_cache_v1
train 42,750 | dev 10,181 | test 13,019 | labels/doc 3.70


## 5. Document view for the decoder: one 1024-token sequence = `head(510)` + cue-selected sentences + `</s>`

In [8]:
MAX_SEG, CUE_CAP, LAST_BONUS_TOKENS = 96, 3, 128

def select_cue_tokens(tok, fl, budget=CONTENT, head_keep=128):
    n = len(tok)
    if n <= budget: return tok
    head = tok[:head_keep]; rest = tok[head_keep:]; rfl = fl[head_keep:]; budget_rest = budget - len(head); nr = len(rest)
    ends = np.flatnonzero(rfl & FLAG_SENT) + 1; bounds = [0] + ends.tolist()
    if bounds[-1] != nr: bounds.append(nr)
    starts_l, ends_l = [], []
    for a, b in zip(bounds[:-1], bounds[1:]):
        while b - a > MAX_SEG: starts_l.append(a); ends_l.append(a + MAX_SEG); a += MAX_SEG
        if b > a: starts_l.append(a); ends_l.append(b)
    starts = np.asarray(starts_l); ends_a = np.asarray(ends_l)
    cs = np.concatenate([[0], np.cumsum((rfl & FLAG_CUE).astype(np.int64))])
    score = np.minimum(cs[ends_a] - cs[starts], CUE_CAP).astype(np.float64)
    score += 0.5 * (starts >= nr - LAST_BONUS_TOKENS); score -= np.arange(len(starts)) * 1e-6
    order = np.argsort(-score, kind="stable"); chosen, used = [], 0
    for k in order:
        s0, e0 = int(starts[k]), int(ends_a[k]); ln = e0 - s0; room = budget_rest - used
        if ln <= room: chosen.append((s0, e0)); used += ln
        elif room >= 16: chosen.append((s0, s0 + room)); used += room
        if budget_rest - used < 8: break
    chosen.sort(); pieces = [rest[s0:e0] for s0, e0 in chosen]
    return np.concatenate([head] + pieces) if pieces else head

def make_view(tok, fl):
    """<= CONTENT tokens: the whole document if it fits, else head(HEAD_KEEP) + cue-selected sentences of the remainder (original order)."""
    if len(tok) <= CONTENT: return tok
    return select_cue_tokens(tok, fl, CONTENT, head_keep=HEAD_KEEP)

class LSIDataset(Dataset):
    def __init__(self, split, indices, with_labels=True):
        self.a = ARR[split]; self.indices = np.asarray(indices, dtype=np.int64); self.with_labels = with_labels
    def __len__(self): return len(self.indices)
    def __getitem__(self, i):
        d = int(self.indices[i]); a = self.a; o0, o1 = int(a["offsets"][d]), int(a["offsets"][d + 1])
        tok = np.asarray(a["tokens"][o0:o1]); fl = np.asarray(a["flags"][o0:o1])
        seq = np.concatenate([make_view(tok, fl).astype(np.int64), [EOS_ID]])         # content + </s>  (no <s>: Phase 1 never used it)
        item = {"seq": seq, "idx": d}
        if self.with_labels: item["y"] = a["labels"][d].astype(np.float32)
        return item

def collate_lsi(batch):
    B = len(batch); T = max(len(b["seq"]) for b in batch)
    ids = torch.full((B, T), PAD_ID, dtype=torch.long)                                 # right padding (causal model: real tokens never see pads)
    for i, b in enumerate(batch): ids[i, :len(b["seq"])] = torch.from_numpy(b["seq"])
    out = {"input_ids": ids, "mask": ids.ne(PAD_ID), "idx": torch.tensor([b["idx"] for b in batch])}
    if "y" in batch[0]: out["y"] = torch.from_numpy(np.stack([b["y"] for b in batch]))
    return out

_demo = LSIDataset("train", [0])[0]
_lens = np.array([len(LSIDataset("train", [i])[0]["seq"]) for i in range(min(200, Ntr))])
print(f"example sequence length {len(_demo['seq'])} (ends with </s>: {_demo['seq'][-1] == EOS_ID}) | first 200 train docs: mean {_lens.mean():.0f}, {100*(_lens == MAX_LEN).mean():.0f}% fill the full {MAX_LEN} tokens")

example sequence length 1024 (ends with </s>: True) | first 200 train docs: mean 747, 34% fill the full 1024 tokens


## 6. Evaluation + post-processing toolkit (same as Exp 1 / Exp 2)

In [9]:
from scipy.ndimage import uniform_filter1d
from sklearn.metrics import average_precision_score

GRID = np.round(np.arange(0.01, 0.96, 0.01), 2)
sigmoid = lambda z: 1.0 / (1.0 + np.exp(-np.clip(z, -30, 30)))

def f1_grid(y, p, grid=GRID):
    y = y.astype(bool); npos = y.sum(0); G, C = len(grid), y.shape[1]
    tp = np.empty((G, C)); fp = np.empty((G, C))
    for gi, g in enumerate(grid):
        pr = p >= g; tp[gi] = (pr & y).sum(0); fp[gi] = pr.sum(0) - tp[gi]
    fn = npos[None, :] - tp
    return 2 * tp / np.maximum(2 * tp + fp + fn, 1)

def best_global(y, p, grid=GRID):
    f1 = f1_grid(y, p, grid); macro = f1.mean(1); i = int(np.argmax(macro)); return float(grid[i]), float(macro[i])

def fit_per_label(y, p, t_glob, min_pos=10, shrink=100.0, grid=GRID):
    f1 = uniform_filter1d(f1_grid(y, p, grid), size=3, axis=0, mode="nearest")
    thr = grid[np.argmax(f1, axis=0)].astype(np.float64); sup = y.sum(0).astype(np.float64)
    thr = np.where(sup < min_pos, t_glob, thr)
    w = sup / (sup + shrink) if shrink > 0 else np.ones_like(sup)
    return w * thr + (1 - w) * t_glob

def predict_binary(p, thr, ensure_one=True):
    pred = p >= thr
    if ensure_one:
        empty = ~pred.any(1)
        if empty.any(): pred[empty, np.argmax(p[empty] / thr[None, :] if np.ndim(thr) else p[empty], axis=1)] = True
    return pred.astype(np.uint8)

def label_f1(y, pred):
    y = y.astype(bool); pr = pred.astype(bool); tp = (y & pr).sum(0); fp = (~y & pr).sum(0); fn = (y & ~pr).sum(0)
    return 2 * tp / np.maximum(2 * tp + fp + fn, 1)

def prf_summary(y, pred):
    yb, pb = y.astype(bool), pred.astype(bool)
    tp = (yb & pb).sum(0); fp = (~yb & pb).sum(0); fn = (yb & ~pb).sum(0)
    prec = tp / np.maximum(tp + fp, 1); rec = tp / np.maximum(tp + fn, 1)
    return dict(macro_f1=float(label_f1(y, pred).mean()), micro_f1=float(2 * tp.sum() / max(2 * tp.sum() + fp.sum() + fn.sum(), 1)),
                macro_precision=float(prec.mean()), macro_recall=float(rec.mean()), subset_acc=float((yb == pb).all(1).mean()),
                avg_pred=float(pb.sum(1).mean()), empty_rate=float((pb.sum(1) == 0).mean()))

def fit_temperature(y, z):
    best, bt = 1e9, 1.0
    for T in np.arange(0.5, 4.01, 0.1):
        p = np.clip(sigmoid(z / T), 1e-7, 1 - 1e-7); l = -(y * np.log(p) + (1 - y) * np.log(1 - p)).mean()
        if l < best: best, bt = l, T
    def tf(z2, T=bt): return z2 / T
    tf.params = {"type": "temperature", "T": float(bt)}; return tf

def fit_platt(y, z, lam=1.0, iters=30):
    C = y.shape[1]; a = np.ones(C); b = np.zeros(C)
    for _ in range(iters):
        p = sigmoid(z * a + b); r = p - y; w = p * (1 - p)
        ga = (r * z).sum(0) + lam * (a - 1); gb = r.sum(0) + lam * b
        haa = (w * z * z).sum(0) + lam; hab = (w * z).sum(0); hbb = w.sum(0) + lam
        det = haa * hbb - hab ** 2 + 1e-12
        a -= (hbb * ga - hab * gb) / det; b -= (-hab * ga + haa * gb) / det; a = np.clip(a, 0.05, 5.0)
    def tf(z2, a=a, b=b): return z2 * a + b
    tf.params = {"type": "platt", "a": a.tolist(), "b": b.tolist()}; return tf

def fit_stacker(y, z, l2=0.01, steps=100, lr=0.02, seed=0):
    N, C = z.shape; a = np.ones(C); b = np.zeros(C); W = np.zeros((C, C)); eye = 1 - np.eye(C)
    params = [a, b, W]; m = [np.zeros_like(q) for q in params]; v = [np.zeros_like(q) for q in params]; zc = np.clip(z, -12, 12)
    for t in range(1, steps + 1):
        logit = zc * a + zc @ W + b; g = (sigmoid(logit) - y) / N
        ga = (g * zc).sum(0); gb = g.sum(0); gW = (zc.T @ g) * eye + l2 * W
        for k, gr in enumerate([ga, gb, gW]):
            m[k] = 0.9 * m[k] + 0.1 * gr; v[k] = 0.999 * v[k] + 0.001 * gr ** 2
            params[k] -= lr * (m[k] / (1 - 0.9 ** t)) / (np.sqrt(v[k] / (1 - 0.999 ** t)) + 1e-8)
        W *= eye
    def tf(z2, a=a, b=b, W=W): return np.clip(z2, -12, 12) * a + np.clip(z2, -12, 12) @ W + b
    tf.params = {"type": "stacker", "a": a.tolist(), "b": b.tolist(), "W": W.tolist()}; return tf

def stack_oof(y, z, l2, K=3):
    idx = np.random.default_rng(1).permutation(len(z)); folds = np.array_split(idx, K); out = np.zeros_like(z)
    for k in range(K):
        te = folds[k]; tr = np.concatenate([folds[j] for j in range(K) if j != k]); out[te] = fit_stacker(y[tr], z[tr], l2)(z[te])
    return out

def make_strategy(calib=None, thr="global", shrink=100.0, min_pos=10, top1=True, l2=0.01):
    def fit(y, z):
        if calib == "temp": tf = fit_temperature(y, z); z_thr = tf(z)
        elif calib == "platt": tf = fit_platt(y, z); z_thr = tf(z)
        elif calib == "stack": tf = fit_stacker(y, z, l2); z_thr = stack_oof(y, z, l2)
        else:
            def tf(q): return q
            tf.params = {"type": "identity"}; z_thr = z
        p_tr = sigmoid(z_thr); tg, _ = best_global(y, p_tr)
        thr_vec = np.full(y.shape[1], tg) if thr == "global" else fit_per_label(y, p_tr, tg, min_pos, shrink)
        def predict(z2): return predict_binary(sigmoid(tf(z2)), thr_vec, top1)
        predict.thr = thr_vec; predict.tf = tf; return predict
    return fit

STRATEGIES = OrderedDict([
    ("global", make_strategy(None, "global", top1=False)), ("global + top1", make_strategy(None, "global")),
    ("per-label raw + top1", make_strategy(None, "label", shrink=0)), ("per-label shrunk(100) + top1", make_strategy(None, "label", shrink=100)),
    ("per-label shrunk(300) + top1", make_strategy(None, "label", shrink=300)), ("temperature -> global", make_strategy("temp", "global")),
    ("Platt -> global", make_strategy("platt", "global")), ("Platt -> per-label shrunk(100)", make_strategy("platt", "label", shrink=100)),
    ("stacker -> global", make_strategy("stack", "global")), ("stacker -> per-label shrunk(100)", make_strategy("stack", "label", shrink=100)),
])

def crossfit(y, z, fit_fn, K=5, seed=0):
    rng = np.random.default_rng(seed); folds = np.array_split(rng.permutation(len(z)), K); pred = np.zeros(y.shape, np.uint8)
    for k in range(K):
        te = folds[k]; tr = np.concatenate([folds[j] for j in range(K) if j != k]); pred[te] = fit_fn(y[tr], z[tr])(z[te])
    return pred

def ece_brier(y, p, bins=15):
    pf, yf = p.ravel(), y.ravel().astype(float); edges = np.linspace(0, 1, bins + 1); ece = 0.0
    idx = np.clip(np.digitize(pf, edges) - 1, 0, bins - 1)
    for b in range(bins):
        m = idx == b
        if m.any(): ece += m.mean() * abs(pf[m].mean() - yf[m].mean())
    return float(ece), float(((pf - yf) ** 2).mean())

def macro_ap(y, p):
    v = [average_precision_score(y[:, j], p[:, j]) for j in range(y.shape[1]) if 0 < y[:, j].sum() < len(y)]
    return float(np.mean(v)) if v else 0.0

def evaluate_logits(y, z, perlabel_cv=True):
    p = sigmoid(z); thr, mf1 = best_global(y, p); pred = (p >= thr).astype(np.uint8)
    out = prf_summary(y, pred); out["best_global_thr"] = thr; out["macro_f1_global"] = mf1
    pc = np.clip(p, 1e-7, 1 - 1e-7); out["bce"] = float(-(y * np.log(pc) + (1 - y) * np.log(1 - pc)).mean()); out["macro_ap"] = macro_ap(y, p)
    if perlabel_cv:
        pr = crossfit(y, z, STRATEGIES["per-label shrunk(100) + top1"], K=4); out["macro_f1_perlabel_cv"] = float(label_f1(y, pr).mean())
    return out

def strategy_table(y, z, K=5):
    rows, preds = [], {}
    for name, fn in STRATEGIES.items():
        t0 = time.time(); pr = crossfit(y, z, fn, K=K); preds[name] = pr
        rows.append(dict(strategy=name, **prf_summary(y, pr), seconds=time.time() - t0))
    return pd.DataFrame(rows).sort_values("macro_f1", ascending=False).reset_index(drop=True), preds
print("toolkit ready")

toolkit ready


## 7. Attention-pooling head over all tokens **including the final `</s>`**, positive-weighted BCE

In [10]:
D_MODEL = cfg.d_model
POOL_INCLUDE_EOS = RECIPE["pool_eos"]

class AttnPoolHead(nn.Module):
    """One learned query attends over every real position of the sequence, the final </s> token included
    (in a causal decoder </s> is the one position that has read the whole document)."""
    def __init__(self, C, dropout, d_att=256):
        super().__init__(); self.proj = nn.Linear(D_MODEL, d_att); self.v = nn.Linear(d_att, 1, bias=False)
        self.drop = nn.Dropout(dropout); self.out = nn.Linear(D_MODEL, C)
    def bias_param(self): return self.out.bias
    def forward(self, h, ids, mask, return_attn=False):
        cm = mask & ids.ne(BOS_ID)
        if not POOL_INCLUDE_EOS: cm = cm & ids.ne(EOS_ID)
        s = self.v(torch.tanh(self.proj(h))).squeeze(-1).float().masked_fill(~cm, -1e4)
        a = torch.softmax(s, dim=1) * cm; a = a / a.sum(1, keepdim=True).clamp_min(1e-9)
        z = self.out(self.drop((a.unsqueeze(-1).to(h.dtype) * h).sum(1)))
        return (z, a) if return_attn else z

class LSIModel(nn.Module):
    def __init__(self, backbone, dropout, prior_logit):
        super().__init__(); self.backbone = backbone; self.head = AttnPoolHead(NUM_LABELS, dropout)
        with torch.no_grad(): self.head.bias_param().copy_(torch.as_tensor(prior_logit, dtype=torch.float32))
    def forward(self, input_ids, mask, return_attn=False):
        h = self.backbone(input_ids).float()
        return self.head(h, input_ids, mask, return_attn)

_pw = torch.as_tensor(np.clip(np.sqrt((Ntr - POS_COUNT) / POS_COUNT), 1, 20), dtype=torch.float32, device=device)
def loss_fn(z, y): return F.binary_cross_entropy_with_logits(z, y, pos_weight=_pw)
print("pos_weight range: %.2f - %.2f | </s> included in pooling: %s" % (_pw.min().item(), _pw.max().item(), POOL_INCLUDE_EOS))

pos_weight range: 1.63 - 12.50 | </s> included in pooling: True


## 8. Training engine for adapter tuning
The whole Phase 1 decoder is frozen, but gradients still flow **through** it to reach the adapters in the lower layers (no weight gradients are computed for frozen weights). Only adapters + head reach the optimizer.

In [13]:
GRAD_CKPT, MB_DOCS, NUM_WORKERS, MAX_GRAD_NORM = False, 8, 2, 1.0
def accum(): return max(1, EFFECTIVE_BATCH // MB_DOCS)

def build_model(grad_ckpt=None):
    bb = LegalBackbone(cfg, grad_ckpt=GRAD_CKPT if grad_ckpt is None else grad_ckpt)
    load_phase1_weights(bb, PHASE1_STATE)
    for p in bb.parameters(): p.requires_grad_(False)                 # freeze the entire Phase 1 decoder
    add_adapters(bb, ADAPTER_BOTTLENECK, ADAPTER_INIT_STD)            # new adapter modules are trainable
    return LSIModel(bb, RECIPE["dropout"], PRIOR_LOGIT).to(device)

def param_report(model):
    tot = sum(p.numel() for p in model.parameters()); tr = sum(p.numel() for p in model.parameters() if p.requires_grad)
    return tot, tr

def trainable_state(model): return {n: p.detach().cpu().clone() for n, p in model.named_parameters() if p.requires_grad}
def load_trainable(model, state):
    names = {n for n, p in model.named_parameters() if p.requires_grad}; assert set(state) == names, "trainable-state keys do not match the model"
    res = model.load_state_dict(state, strict=False); assert not res.unexpected_keys

def make_optimizer(model):
    groups = {}
    for name, p in model.named_parameters():
        if not p.requires_grad: continue
        lr = RECIPE["head_lr"] if name.startswith("head.") else RECIPE["adapter_lr"]
        wd = 0.0 if p.ndim <= 1 else RECIPE["wd"]
        groups.setdefault((lr, wd), []).append(p)
    return torch.optim.AdamW([{"params": ps, "lr": lr, "weight_decay": wd} for (lr, wd), ps in groups.items()], betas=(0.9, 0.999))

def build_training(model, n_docs, epochs):
    mb, acc = MB_DOCS, accum(); spe = math.ceil(math.ceil(n_docs / mb) / acc); total = spe * epochs; warm = max(1, int(total * RECIPE["warmup"]))
    lam = lambda s: (s + 1) / warm if s < warm else max(0.0, (total - s) / max(1, total - warm))
    opt = make_optimizer(model)
    return dict(opt=opt, sched=torch.optim.lr_scheduler.LambdaLR(opt, lam), scaler=torch.amp.GradScaler("cuda"), mb=mb, accum=acc, total_steps=total)

def make_loader(ds, batch, shuffle, seed=0):
    g = torch.Generator(); g.manual_seed(seed)
    return DataLoader(ds, batch_size=batch, shuffle=shuffle, generator=g if shuffle else None, collate_fn=collate_lsi, num_workers=NUM_WORKERS, pin_memory=True)

def to_dev(b): return b["input_ids"].to(device, non_blocking=True), b["mask"].to(device, non_blocking=True)

def train_one_epoch(model, loader, T, tag="", log_frac=0.1):
    model.train(); opt, sched, scaler, acc = T["opt"], T["sched"], T["scaler"], T["accum"]
    trainable = [p for p in model.parameters() if p.requires_grad]
    opt.zero_grad(set_to_none=True); tot, nd, t0, n_micro = 0.0, 0, time.perf_counter(), len(loader); next_log = max(1, int(n_micro * log_frac))
    for i, b in enumerate(loader, 1):
        ids, mask = to_dev(b); y = b["y"].to(device, non_blocking=True)
        with torch.autocast("cuda", dtype=torch.float16): logits = model(ids, mask)
        loss = loss_fn(logits.float(), y)
        if not torch.isfinite(loss): raise RuntimeError(f"non-finite loss at micro-batch {i}")
        scaler.scale(loss / acc).backward(); tot += loss.item() * ids.shape[0]; nd += ids.shape[0]
        if i % acc == 0 or i == n_micro:
            scaler.unscale_(opt); torch.nn.utils.clip_grad_norm_(trainable, MAX_GRAD_NORM)
            scaler.step(opt); scaler.update(); opt.zero_grad(set_to_none=True); sched.step()
        if i >= next_log or i == n_micro:
            next_log += max(1, int(n_micro * log_frac)); print(f"   {tag} {i}/{n_micro} | loss {tot/nd:.4f} | {nd/(time.perf_counter()-t0):.1f} docs/s", flush=True)
    return tot / max(nd, 1), nd / (time.perf_counter() - t0)

@torch.inference_mode()
def predict_logits(model, ds, batch_docs):
    model.eval(); out = []; t0 = time.perf_counter()
    for b in make_loader(ds, batch_docs, False):
        ids, mask = to_dev(b)
        with torch.autocast("cuda", dtype=torch.float16): out.append(model(ids, mask).float().cpu().numpy())
    return np.concatenate(out), len(ds) / (time.perf_counter() - t0)

# ---- auto-tune micro-batch / gradient checkpointing (worst case = full 1024-token sequences)
def autotune_memory():
    global GRAD_CKPT, MB_DOCS
    res = []
    for ckpt, docs in [(False, 8), (False, 4), (True, 16),(False,16), (True, 8), (True, 4), (True, 2),(True, 24),(False,24)]:
        model = opt = scaler = None
        try:
            torch.cuda.empty_cache(); torch.cuda.reset_peak_memory_stats()
            model = build_model(grad_ckpt=ckpt); model.train(); opt = make_optimizer(model); scaler = torch.amp.GradScaler("cuda")
            ids = torch.randint(4, 50_000, (docs, MAX_LEN), device=device); ids[:, -1] = EOS_ID; mask = torch.ones_like(ids, dtype=torch.bool)
            y = (torch.rand(docs, NUM_LABELS, device=device) < 0.03).float()
            for step in range(4):
                if step == 1: torch.cuda.synchronize(); t0 = time.perf_counter()
                with torch.autocast("cuda", dtype=torch.float16): lg = model(ids, mask)
                loss = loss_fn(lg.float(), y); opt.zero_grad(set_to_none=True); scaler.scale(loss).backward(); scaler.step(opt); scaler.update()
            torch.cuda.synchronize(); dps = docs * 3 / (time.perf_counter() - t0)
            res.append((dps, ckpt, docs, torch.cuda.max_memory_allocated() / 1024**3)); print(f"grad_ckpt={ckpt!s:5} docs/micro-batch={docs:2d}: {dps:5.1f} doc/s | peak {res[-1][3]:.1f} GiB")
        except torch.cuda.OutOfMemoryError:
            print(f"grad_ckpt={ckpt!s:5} docs/micro-batch={docs:2d}: OOM")
        finally:
            del model, opt, scaler; gc.collect(); torch.cuda.empty_cache()
    if not res: raise RuntimeError("Nothing fits in GPU memory.")
    dps, GRAD_CKPT, MB_DOCS, _ = max(res); print(f"-> grad_ckpt={GRAD_CKPT}, {MB_DOCS} documents/micro-batch ({dps:.1f} doc/s)"); return dps

DOCS_PER_S = autotune_memory()
print(f"estimated one full-train epoch: {Ntr / DOCS_PER_S / 60:.0f} min of training + dev evaluation (Exp 1 full fine-tuning: ~57 min incl. dev eval)")

grad_ckpt=False docs/micro-batch= 8:  13.2 doc/s | peak 6.6 GiB
grad_ckpt=False docs/micro-batch= 4:  12.7 doc/s | peak 4.0 GiB
grad_ckpt=True  docs/micro-batch=16:   9.6 doc/s | peak 2.8 GiB
grad_ckpt=False docs/micro-batch=16:  13.2 doc/s | peak 11.9 GiB
grad_ckpt=True  docs/micro-batch= 8:   9.3 doc/s | peak 2.0 GiB
grad_ckpt=True  docs/micro-batch= 4:   9.0 doc/s | peak 1.5 GiB
grad_ckpt=True  docs/micro-batch= 2:   8.4 doc/s | peak 1.3 GiB
grad_ckpt=True  docs/micro-batch=24:   9.4 doc/s | peak 3.6 GiB
grad_ckpt=False docs/micro-batch=24: OOM
-> grad_ckpt=False, 8 documents/micro-batch (13.2 doc/s)
estimated one full-train epoch: 54 min of training + dev evaluation (Exp 1 full fine-tuning: ~57 min incl. dev eval)


## 9. FINAL TRAINING — adapters + head, full train set, 6 epochs, patience 2

In [14]:
FINAL_BEST_PT, FINAL_LAST_PT = RUN_DIR / "final_best.pt", RUN_DIR / "final_last.pt"
FINAL_DEV_LOGITS = RUN_DIR / "final_best_dev_logits.npy"; FINAL_HISTORY = RUN_DIR / "final_history.json"

final_train_idx = np.arange(Ntr) if FINAL_TRAIN_DOCS_LIMIT is None else np.sort(np.random.default_rng(SEED).permutation(Ntr)[:FINAL_TRAIN_DOCS_LIMIT])
final_dev_idx = np.arange(Ndv) if FINAL_TRAIN_DOCS_LIMIT is None else np.arange(min(Ndv, 512))
Y_final_dev = Ydv[final_dev_idx]

seed_everything(SEED)
final_model = build_model()
n_total, n_train = param_report(final_model)
assert n_train == adapter_param_count(cfg.d_model, cfg.n_layers, ADAPTER_BOTTLENECK) + sum(p.numel() for p in final_model.head.parameters()), "trainable parameter count != hand formula"
print(f"parameters: total {n_total/1e6:.2f} M | trainable {n_train/1e6:.2f} M ({100*n_train/n_total:.2f} %) | frozen {(n_total-n_train)/1e6:.1f} M")
FT = build_training(final_model, len(final_train_idx), FINAL_MAX_EPOCHS)
final_train_ds = LSIDataset("train", final_train_idx); final_dev_ds = LSIDataset("dev", final_dev_idx)
torch.cuda.reset_peak_memory_stats()

history, best_f1, best_epoch, bad, start_epoch = [], -1.0, 0, 0, 1
if RESUME_FINAL and FINAL_LAST_PT.exists():
    ck = torch.load(FINAL_LAST_PT, map_location="cpu", weights_only=False)
    load_trainable(final_model, ck["trainable"]); FT["opt"].load_state_dict(ck["opt"]); FT["sched"].load_state_dict(ck["sched"]); FT["scaler"].load_state_dict(ck["scaler"])
    history, best_f1, best_epoch, bad, start_epoch = ck["history"], ck["best_f1"], ck["best_epoch"], ck["bad"], ck["epoch"] + 1
    del ck; print(f"resumed after epoch {start_epoch-1}; best so far {best_f1:.4f} (epoch {best_epoch})")

run_start = time.time()
for epoch in range(start_epoch, FINAL_MAX_EPOCHS + 1):
    ep0 = time.time()
    tr_loss, tr_dps = train_one_epoch(final_model, make_loader(final_train_ds, FT["mb"], True, SEED + epoch), FT, tag=f"adapter ep{epoch}/{FINAL_MAX_EPOCHS}")
    z_dev, eval_dps = predict_logits(final_model, final_dev_ds, FT["mb"] * 2); m = evaluate_logits(Y_final_dev, z_dev)
    rec = dict(epoch=epoch, train_loss=tr_loss, train_docs_per_s=tr_dps, **{k: m[k] for k in ["macro_f1_global", "best_global_thr", "macro_ap", "macro_f1_perlabel_cv", "micro_f1", "subset_acc", "macro_precision", "macro_recall", "bce"]}, epoch_minutes=(time.time() - ep0) / 60)
    history.append(rec)
    print(f"\n=== epoch {epoch}: train loss {tr_loss:.4f} | dev macro-F1 {m['macro_f1_global']:.4f} @thr {m['best_global_thr']:.2f} | per-label(CV) {m['macro_f1_perlabel_cv']:.4f} | macro-AP {m['macro_ap']:.4f} | micro-F1 {m['micro_f1']:.4f} | {rec['epoch_minutes']:.1f} min ===\n")
    if m["macro_f1_global"] > best_f1 + 1e-4:
        best_f1, best_epoch, bad = m["macro_f1_global"], epoch, 0
        torch.save({"trainable": trainable_state(final_model), "epoch": epoch, "dev_macro_f1_global": best_f1}, FINAL_BEST_PT)
        np.save(FINAL_DEV_LOGITS, z_dev.astype(np.float32)); print(f"   new best epoch {epoch} saved (adapters + head only)")
    else:
        bad += 1; print(f"   no improvement ({bad}/{FINAL_PATIENCE})")
    json.dump(history, open(FINAL_HISTORY, "w"), indent=1)
    torch.save({"trainable": trainable_state(final_model), "opt": FT["opt"].state_dict(), "sched": FT["sched"].state_dict(), "scaler": FT["scaler"].state_dict(), "history": history, "best_f1": best_f1, "best_epoch": best_epoch, "bad": bad, "epoch": epoch}, FINAL_LAST_PT)
    if bad >= FINAL_PATIENCE: print("early stopping."); break
train_peak_gib = torch.cuda.max_memory_allocated() / 1024**3
print(f"\nfinished: best epoch {best_epoch}, dev macro-F1 {best_f1:.4f} | wall {(time.time()-run_start)/3600:.2f} h | peak GPU {train_peak_gib:.1f} GiB")
print(f"Exp 1 (encoder + full fine-tuning) reference dev macro-F1: {EXP1_DEV_MACRO_F1:.4f}  ->  SWA + adapters {best_f1:.4f} ({best_f1-EXP1_DEV_MACRO_F1:+.4f})")
display(pd.DataFrame(history).round(4))

parameters: total 140.95 M | trainable 1.86 M (1.32 %) | frozen 139.1 M
   adapter ep1/6 534/5344 | loss 0.4759 | 13.0 docs/s
   adapter ep1/6 1068/5344 | loss 0.4204 | 13.0 docs/s
   adapter ep1/6 1602/5344 | loss 0.3922 | 13.0 docs/s
   adapter ep1/6 2136/5344 | loss 0.3763 | 13.0 docs/s
   adapter ep1/6 2670/5344 | loss 0.3648 | 13.0 docs/s
   adapter ep1/6 3204/5344 | loss 0.3568 | 13.0 docs/s
   adapter ep1/6 3738/5344 | loss 0.3503 | 13.0 docs/s
   adapter ep1/6 4272/5344 | loss 0.3445 | 13.0 docs/s
   adapter ep1/6 4806/5344 | loss 0.3398 | 13.0 docs/s
   adapter ep1/6 5340/5344 | loss 0.3357 | 13.0 docs/s
   adapter ep1/6 5344/5344 | loss 0.3357 | 13.0 docs/s

=== epoch 1: train loss 0.3357 | dev macro-F1 0.3152 @thr 0.47 | per-label(CV) 0.3337 | macro-AP 0.2887 | micro-F1 0.4154 | 59.9 min ===

   new best epoch 1 saved (adapters + head only)
   adapter ep2/6 534/5344 | loss 0.2838 | 13.0 docs/s
   adapter ep2/6 1068/5344 | loss 0.2842 | 13.0 docs/s
   adapter ep2/6 1602/5344 

,epoch,train_loss,train_docs_per_s,macro_f1_global,best_global_thr,macro_ap,macro_f1_perlabel_cv,micro_f1,subset_acc,macro_precision,macro_recall,bce,epoch_minutes
0,1,0.3357,13.0259,0.3152,0.47,0.2887,0.3337,0.4154,0.0139,0.2894,0.4125,0.1542,59.8610
1,2,0.2773,13.0367,0.3465,0.51,0.3253,0.3587,0.4407,0.0323,0.3215,0.4260,0.1469,59.8248
2,3,0.2467,13.0313,0.3657,0.48,0.3439,0.3770,0.4586,0.0348,0.3505,0.4246,0.1284,59.8449
3,4,0.2125,13.0275,0.3806,0.61,0.3511,0.3874,0.4667,0.0469,0.3772,0.4104,0.1359,59.8630
4,5,0.1720,13.0304,0.3845,0.55,0.3531,0.3894,0.4750,0.0535,0.3631,0.4203,0.1253,59.8455
5,6,0.1334,13.0213,0.3813,0.56,0.3458,0.3866,0.4727,0.0542,0.3711,0.4001,0.1251,59.8744


## 10. Best epoch → post-processing chosen out-of-fold on dev → test predictions → submission + final `.pt`

In [15]:
ck = torch.load(FINAL_BEST_PT, map_location="cpu", weights_only=False)
load_trainable(final_model, ck["trainable"]); final_model.eval(); best_epoch = ck["epoch"]; best_trainable = ck["trainable"]; del ck
z_dev_final = np.load(FINAL_DEV_LOGITS); assert z_dev_final.shape == Y_final_dev.shape

POST_TABLE, POST_PREDS = strategy_table(Y_final_dev, z_dev_final, K=5)
display(POST_TABLE.round(4))
POST_NAME = POST_TABLE.iloc[0]["strategy"]
post_predictor = STRATEGIES[POST_NAME](Y_final_dev, z_dev_final); dev_oof_pred = POST_PREDS[POST_NAME]
base_row = POST_TABLE[POST_TABLE.strategy == "global"].iloc[0]
print(f"chosen post-processing: {POST_NAME} | OOF dev macro-F1 {POST_TABLE.iloc[0]['macro_f1']:.4f} | micro-F1 {POST_TABLE.iloc[0]['micro_f1']:.4f} | subset acc {POST_TABLE.iloc[0]['subset_acc']:.4f} (plain global: {base_row['macro_f1']:.4f})")

test_ds = LSIDataset("test", np.arange(Nte), with_labels=False)
t0 = time.time(); z_test, test_dps = predict_logits(final_model, test_ds, FT["mb"] * 2); infer_s = time.time() - t0
np.save(RUN_DIR / "test_logits.npy", z_test.astype(np.float32)); print(f"test inference: {Nte:,} docs in {infer_s/60:.1f} min ({test_dps:.1f} docs/s)")
test_pred = post_predictor(z_test)

def write_submission(case_ids, pred_bin, path):
    out = [{"id": cid, "pred": sorted({int(j) for j in np.flatnonzero(pred_bin[i])})} for i, cid in enumerate(case_ids)]
    assert len(out) == len(case_ids) == len(set(map(str, case_ids)))
    for o in out: assert all(isinstance(x, int) and 0 <= x < NUM_LABELS for x in o["pred"]) and len(o["pred"]) == len(set(o["pred"]))
    json.dump(out, open(path, "w")); return out

test_ids = ARR["test"]["ids"]
submission = write_submission(test_ids, test_pred, RUN_DIR / "submission.json")
if ON_KAGGLE: write_submission(test_ids, test_pred, WORK / "submission_exp3_swa_adapter.json")
n_lab = np.array([len(o["pred"]) for o in submission])
print(f"submission: {len(submission):,} objects | labels/doc mean {n_lab.mean():.2f}, min {n_lab.min()}, max {n_lab.max()} | empty {int((n_lab==0).sum())}")

post = dict(name=POST_NAME, thresholds=np.asarray(post_predictor.thr).tolist(), calibrator=post_predictor.tf.params, ensure_at_least_one_label=True)
meta = {"recipe": dict(RECIPE), "adapter": {"bottleneck": ADAPTER_BOTTLENECK, "init_std": ADAPTER_INIT_STD, "placement": "after every transformer block output", "n_adapters": cfg.n_layers, "backbone_frozen": True},
        "decoder_config": asdict(cfg), "num_labels": NUM_LABELS, "best_epoch": best_epoch, "dev_metrics_best_epoch": history[best_epoch - 1] if history else None, "postprocess": post,
        "dev_oof_macro_f1_after_postprocess": float(POST_TABLE.iloc[0]["macro_f1"]), "max_seq_len": MAX_LEN, "content_tokens": CONTENT, "head_keep": HEAD_KEEP, "appends_eos": True, "prepends_bos": False,
        "tokenizer_path": str(PHASE1_TOKENIZER_PATH), "phase1_checkpoint": str(PHASE1_CHECKPOINT_PATH), "experiment_3": "phase1 SWA decoder + bottleneck adapters"}
ADAPTER_ONLY_PT = RUN_DIR / "lsi_swa_adapters_only.pt"
torch.save({**meta, "trainable_state": best_trainable}, ADAPTER_ONLY_PT)                      # small file: adapters + head (load on top of the Phase 1 checkpoint)
TRAINABLE_BYTES = sum(v.numel() * v.element_size() for v in best_trainable.values())
FINAL_PT = RUN_DIR / "lsi_swa_adapter_final.pt"
torch.save({**meta, "model_state_dict": {k: v.detach().cpu() for k, v in final_model.state_dict().items()}}, FINAL_PT)   # self-contained: frozen backbone + adapters + head
gc.collect()
print(f"saved {FINAL_PT} ({FINAL_PT.stat().st_size/1024**2:.0f} MB, self-contained) and {ADAPTER_ONLY_PT} ({ADAPTER_ONLY_PT.stat().st_size/1024**2:.1f} MB, adapters + head only)")

,strategy,macro_f1,micro_f1,macro_precision,macro_recall,subset_acc,avg_pred,empty_rate,seconds
0,per-label shrunk(100) + top1,0.3900,0.4823,0.3678,0.4265,0.0486,4.9019,0.0000,2.0478
1,stacker -> per-label shrunk(100),0.3897,0.4911,0.3890,0.4058,0.0501,4.7802,0.0000,36.5394
2,per-label raw + top1,0.3893,0.4802,0.3831,0.4184,0.0480,4.9589,0.0000,2.0153
3,per-label shrunk(300) + top1,0.3890,0.4826,0.3630,0.4289,0.0495,4.8256,0.0000,2.0685
4,Platt -> per-label shrunk(100),0.3876,0.4894,0.3911,0.4016,0.0503,4.7733,0.0000,4.6644
5,temperature -> global,0.3847,0.4749,0.3630,0.4207,0.0568,4.2756,0.0000,3.3989
6,global,0.3836,0.4741,0.3622,0.4192,0.0536,4.2706,0.0489,1.0362
7,global + top1,0.3835,0.4739,0.3601,0.4218,0.0557,4.3195,0.0000,1.0250
8,stacker -> global,0.3816,0.4918,0.3702,0.4194,0.0350,5.5657,0.0000,32.6984
9,Platt -> global,0.3769,0.4859,0.3741,0.4177,0.0343,5.8210,0.0000,3.6673


chosen post-processing: per-label shrunk(100) + top1 | OOF dev macro-F1 0.3900 | micro-F1 0.4823 | subset acc 0.0486 (plain global: 0.3836)
test inference: 13,019 docs in 6.6 min (33.1 docs/s)
submission: 13,019 objects | labels/doc mean 4.77, min 1, max 32 | empty 0
saved /kaggle/working/lsi_exp3_swa_adapter/lsi_swa_adapter_final.pt (538 MB, self-contained) and /kaggle/working/lsi_exp3_swa_adapter/lsi_swa_adapters_only.pt (7.1 MB, adapters + head only)


## 11. Analysis for the report (statute frequency, confusions, adapter-layer ablation, `</s>` attention, cost)

In [16]:
y, pr = Y_final_dev, dev_oof_pred
f1 = label_f1(y, pr); sup_tr = Ytr.sum(0); sup_dv = y.sum(0); yb, pb = y.astype(bool), pr.astype(bool)
tp = (yb & pb).sum(0); fp = (~yb & pb).sum(0); fn = (yb & ~pb).sum(0)
per_label = pd.DataFrame({"statute": np.arange(NUM_LABELS), "train_support": sup_tr, "dev_support": sup_dv, "precision": tp / np.maximum(tp + fp, 1), "recall": tp / np.maximum(tp + fn, 1), "f1": f1, "FP": fp, "FN": fn})
per_label.to_csv(RUN_DIR / "swa_adapter_per_label_metrics.csv", index=False)
per_label["support_quartile"] = pd.qcut(per_label["train_support"].rank(method="first"), 4, labels=["rarest 25%", "rare-mid", "common-mid", "most common 25%"])
print("F1 by statute frequency:"); display(per_label.groupby("support_quartile", observed=True)[["f1", "precision", "recall"]].mean().round(3))
print("worst statutes:"); display(per_label.sort_values("f1").head(10).round(3).reset_index(drop=True))
print("best statutes:"); display(per_label.sort_values("f1", ascending=False).head(8).round(3).reset_index(drop=True))
print("corr(log train support, F1) = %.3f" % np.corrcoef(np.log(sup_tr), f1)[0, 1])

M = (yb & ~pb).T.astype(np.int64) @ (~yb & pb).astype(np.int64); np.fill_diagonal(M, 0)
top = np.dstack(np.unravel_index(np.argsort(-M.ravel())[:10], M.shape))[0]
print("\nmost common 'missed j / predicted-instead k' pairs:")
for j, k in top: print(f"  missed {j:>2d} / predicted {k:>2d}: {M[j,k]:4d} docs")

for name, csv in [("Exp 1 full-FT", WORK / "lsi_runs" / "final_per_label_metrics.csv"), ("Exp 2 partial-FT", WORK / "lsi_exp2_partial" / "partial_per_label_metrics.csv")]:
    if csv.exists():
        e = pd.read_csv(csv); print(f"per-label F1: SWA+adapters {per_label['f1'].mean():.4f} vs {name} {e['f1'].mean():.4f} | adapters better on {(per_label['f1'].values > e['f1'].values).sum()} / {NUM_LABELS} statutes")

# ---- where does the pooling head look? attention mass on the final </s> token and on the head vs cue-selected part
def attention_report(n_docs=min(300, len(final_dev_ds))):
    final_model.eval(); eos_mass, head_mass, cue_mass, uni = [], [], [], []
    for b in make_loader(torch.utils.data.Subset(final_dev_ds, range(n_docs)), 8, False):
        ids, mask = to_dev(b)
        with torch.inference_mode(), torch.autocast("cuda", dtype=torch.float16): _, a = final_model(ids, mask, return_attn=True)
        a = a.float().cpu().numpy(); idn = ids.cpu().numpy()
        for i in range(len(a)):
            L = int(mask[i].sum().item()); eos_mass.append(a[i, L - 1]); uni.append(1.0 / L)
            if L > HEAD_KEEP + 1: head_mass.append(a[i, :HEAD_KEEP].sum()); cue_mass.append(a[i, HEAD_KEEP:L - 1].sum())
    print(f"attention-pooling mass on {n_docs} dev docs: </s> token {np.mean(eos_mass):.3f} (uniform attention would give {np.mean(uni):.4f}) | first {HEAD_KEEP} tokens {np.mean(head_mass):.3f} | cue-selected remainder {np.mean(cue_mass):.3f} (long docs only)")
attention_report()

# ---- adapter layer ablation (no retraining): switch adapters off by layer group, measure dev macro-F1 at each configuration's best global threshold
ABL_ROWS = None
if RUN_ADAPTER_ABLATION:
    abl_idx = np.sort(np.random.default_rng(SEED).permutation(Ndv)[:ABLATION_DOCS]) if Ndv > ABLATION_DOCS else np.arange(Ndv)
    abl_ds, Y_abl = LSIDataset("dev", abl_idx), Ydv[abl_idx]; NL = cfg.n_layers
    def set_on(on_layers):
        for i, blk in enumerate(final_model.backbone.blocks): blk.adapter.enabled = i in on_layers
    def abl_score(off_layers):
        set_on(set(range(NL)) - set(off_layers)); z, _ = predict_logits(final_model, abl_ds, FT["mb"] * 2); return z, best_global(Y_abl, sigmoid(z))[1]
    z_intact, f_intact = abl_score([]); rows = [("intact (all 16 adapters on)", f_intact)]
    groups = [("lower half off (layers 1-8)", range(0, NL // 2)), ("upper half off (layers 9-16)", range(NL // 2, NL)), ("ALL adapters off (= frozen Phase 1 + trained head)", range(NL))]
    groups += [(f"layers {s+1}-{s+4} off", range(s, s + 4)) for s in range(0, NL, 4)] + [(f"layer {i+1} off", [i]) for i in range(NL)]
    for name, layers in groups: rows.append((name, abl_score(list(layers))[1]))
    z_back, _ = abl_score([]); restore_err = float(np.abs(z_back - z_intact).max()); set_on(set(range(NL)))
    ABL_ROWS = pd.DataFrame(rows, columns=["configuration", "macro_f1"]); ABL_ROWS["drop_vs_intact"] = f_intact - ABL_ROWS["macro_f1"]
    ABL_ROWS.to_csv(RUN_DIR / "adapter_layer_ablation.csv", index=False)
    print(f"\nadapter ablation on {len(abl_idx)} dev docs (re-enabling all adapters restores the logits: max |diff| {restore_err:.2e})"); display(ABL_ROWS.round(4))

print("\n================ COST SUMMARY (Exp 3: SWA decoder + adapters) ================")
print(f"trainable params: {n_train/1e6:.2f} M of {n_total/1e6:.2f} M ({100*n_train/n_total:.2f} %) | trainable-state size {TRAINABLE_BYTES/1024**2:.1f} MB (full fine-tuning would store ~{BACKBONE_PARAMS*4/1024**2:.0f} MB)")
print(f"inference params: {n_total/1e6:.2f} M (+{adapter_param_count(cfg.d_model, cfg.n_layers, ADAPTER_BOTTLENECK)/1e6:.2f} M adapters over the frozen decoder, not mergeable into the weights)")
print(f"train throughput: {np.mean([h['train_docs_per_s'] for h in history]):.1f} docs/s | epoch time {np.mean([h['epoch_minutes'] for h in history]):.1f} min (Exp 1: ~13.5 docs/s, 56.8 min) | peak train GPU {train_peak_gib:.1f} GiB (Exp 1: 7.7 GiB)")
print(f"test inference: {test_dps:.1f} docs/s at {MAX_LEN} tokens/doc | grad checkpointing {GRAD_CKPT} | micro-batch docs {MB_DOCS}")
print(f"best epoch {best_epoch} | dev macro-F1 (global thr) {best_f1:.4f} | after post-processing (OOF) {POST_TABLE.iloc[0]['macro_f1']:.4f} | Exp 1 reference {EXP1_DEV_MACRO_F1:.4f}")

F1 by statute frequency:


,f1,precision,recall
support_quartile,,,
rarest 25%,0.290,0.275,0.322
rare-mid,0.363,0.348,0.390
common-mid,0.398,0.375,0.433
most common 25%,0.509,0.473,0.561


worst statutes:


,statute,train_support,dev_support,precision,recall,f1,FP,FN,support_quartile
0,8,349,92,0.107,0.130,0.118,100,80,rarest 25%
1,0,343,86,0.098,0.163,0.122,129,72,rarest 25%
2,52,1084,272,0.111,0.165,0.133,360,227,common-mid
3,2,367,98,0.139,0.143,0.141,87,84,rarest 25%
4,76,530,142,0.132,0.190,0.156,178,115,rare-mid
5,40,543,152,0.129,0.197,0.156,203,122,rare-mid
6,82,604,156,0.126,0.224,0.162,242,121,rare-mid
7,99,674,176,0.192,0.142,0.163,105,151,rare-mid
8,32,385,99,0.164,0.253,0.199,127,74,rarest 25%
9,98,490,123,0.179,0.252,0.209,142,92,rare-mid


best statutes:


,statute,train_support,dev_support,precision,recall,f1,FP,FN,support_quartile
0,28,602,151,0.809,0.841,0.825,30,24,rare-mid
1,34,10380,2596,0.801,0.818,0.809,528,472,most common 25%
2,20,487,122,0.699,0.836,0.761,44,20,rare-mid
3,81,1321,346,0.785,0.728,0.756,69,94,common-mid
4,94,3835,958,0.717,0.703,0.710,265,285,most common 25%
5,29,497,125,0.704,0.648,0.675,34,44,rare-mid
6,36,569,142,0.617,0.648,0.632,57,50,rare-mid
7,61,3723,932,0.647,0.618,0.632,314,356,most common 25%


corr(log train support, F1) = 0.537

most common 'missed j / predicted-instead k' pairs:
  missed 14 / predicted  5:  193 docs
  missed 13 / predicted  5:  178 docs
  missed 12 / predicted  5:  173 docs
  missed 39 / predicted 43:  157 docs
  missed 39 / predicted  5:  146 docs
  missed 77 / predicted 43:  130 docs
  missed 39 / predicted 97:  128 docs
  missed  5 / predicted 43:  127 docs
  missed 34 / predicted 43:  123 docs
  missed  5 / predicted 97:  122 docs
attention-pooling mass on 300 dev docs: </s> token 0.003 (uniform attention would give 0.0029) | first 510 tokens 0.358 | cue-selected remainder 0.641 (long docs only)

adapter ablation on 600 dev docs (re-enabling all adapters restores the logits: max |diff| 0.00e+00)


,configuration,macro_f1,drop_vs_intact
0,intact (all 16 adapters on),0.3795,0.0000
1,lower half off (layers 1-8),0.2731,0.1064
2,upper half off (layers 9-16),0.2999,0.0796
3,ALL adapters off (= frozen Phase 1 + trained h...,0.1807,0.1988
4,layers 1-4 off,0.3102,0.0692
5,layers 5-8 off,0.3486,0.0309
6,layers 9-12 off,0.3406,0.0388
7,layers 13-16 off,0.3618,0.0176
8,layer 1 off,0.3611,0.0183
9,layer 2 off,0.3722,0.0072



================ COST SUMMARY (Exp 3: SWA decoder + adapters) ================
trainable params: 1.86 M of 140.95 M (1.32 %) | trainable-state size 7.1 MB (full fine-tuning would store ~531 MB)
inference params: 140.95 M (+1.59 M adapters over the frozen decoder, not mergeable into the weights)
train throughput: 13.0 docs/s | epoch time 59.9 min (Exp 1: ~13.5 docs/s, 56.8 min) | peak train GPU 6.6 GiB (Exp 1: 7.7 GiB)
test inference: 33.1 docs/s at 1024 tokens/doc | grad checkpointing False | micro-batch docs 8
best epoch 5 | dev macro-F1 (global thr) 0.3845 | after post-processing (OOF) 0.3900 | Exp 1 reference 0.3826
